## Meta Ads

#### Useful links:
- [Get a Developer account](https://developers.facebook.com/async/registration/dialog/?src=default)
- [Identity and Location Confirmation](Facebook.com/ID)
- [Meta Ads API Documentation](https://www.facebook.com/ads/library/api/)
- [Meta Ads API - Ads Archive Reference](https://developers.facebook.com/docs/graph-api/reference/ads_archive/)


In [1]:
# requirements
%pip install pandas==2.3.2 requests==2.32.5 python-dotenv==1.1.1 pandera==0.26.1

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import glob
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaErrors
import requests
from dotenv import load_dotenv
from datetime import datetime
from time import sleep

In [2]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.11.10 | packaged by Anaconda, Inc. | (main, Oct  3 2024, 07:22:26) [MSC v.1929 64 bit (AMD64)]


In [3]:
# environment variables
load_dotenv()
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN") # Token expires in
FILEPATH = os.getenv("FILEPATH")

#### Meta Ads API Available Parameters


- ad_active_status: *enum {ACTIVE, ALL, INACTIVE}* <span style="color:red">**(DEFAULT: Active)**</span>
- ad_delivery_date_max: *str*
- ad_delivery_date_min: *str*
- ad_reached_countries: *array<enum> (countries abbreviations)*
- ad_type: *enum {ALL, EMPLOYMENT_ADS, FINANCIAL_PRODUCTS_AND_SERVICES_ADS, HOUSING_ADS, POLITICAL_AND_ISSUE_ADS}* <span style="color:red">**(DEFAULT: ALL)**</span>
- bylines: *array<str>*
- delivery_by_region: *array<str>*
- estimated_audience_size_max: *int64*
- estimated_audience_size_min: *int64*
- languages: *array<str>*
- media_type: *enum {ALL, IMAGE, MEME, VIDEO, NONE}*
- publisher_platforms: *array<enum {FACEBOOK, INSTAGRAM, AUDIENCE_NETWORK, MESSENGER, WHATSAPP, OCULUS, THREADS}>*
- search_page_ids: *array<int64>*
- search_terms: *str* <span style="color:red">**(DEFAULT: "")**</span>
- search_type: *enum {KEYWORD_UNORDERED, KEYWORD_EXACT_PHRASE}* <span style="color:red">**(DEFAULT: KEYWORD_UNORDERED)**</span>
- unmask_removed_content: *bool* <span style="color:red">**(DEFAULT: False)**</span>

In [4]:
# Expected Payload
# It consists of a dict containing the access token and the desired parameters for the request
# Must have search_terms or search_page_ids parameters
# Please update the variable below with the desired search parameters
payload = {
            "access_token": ACCESS_TOKEN,
        }

#### Meta Ads API Available Fields

- All countries:
    * id
    * ad_creation_time
    * ad_creative_bodies
    * ad_creative_link_captions
    * ad_creative_link_descriptions
    * ad_creative_link_titles
    * ad_delivery_start_time
    * ad_delivery_stop_time
    * ad_snapshot_url
    * bylines
    * currency
    * delivery_by_region
    * demographic_distribution
    * estimated_audience_size
    * impressions
    * languages
    * page_id
    * page_name
    * publisher_platforms
    * spend
 
- Brazil only:
    * age_country_gender_reach_breakdown
    * br_total_reach
    * target_ages
    * target_gender
    * target_locations
    * total_reach_by_location
    
- EU only:
    * age_country_gender_reach_breakdown
    * beneficiary_payers
    * eu_total_reach
    * target_ages
    * target_gender
    * target_locations
    * total_reach_by_location


#### API Connection

In [5]:
# fields should be a string of selected fields separeted by comma, without any blank space
# e.g: "id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions"
fields = ""

In [6]:
# newest api version is v23.0 (08-09-2025)
api_url = f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"

In [7]:
# API Requests
def request_to_api(session, payload, api_url):
    response = session.get(api_url, params=payload)
    return response.json()

# Pagination
def paginate(session, payload, response):
    collected_ads = []
    while "paging" in response:
        next_page_url = response["paging"].get("next", "")
        sleep(1)
        response = request_to_api(session,payload, next_page_url)
        # The implementation of pagination ends here
        # Feel free to change implementation to extract data
        data = response.get("data", [])
        collected_ads.extend(data)
    return collected_ads

# Session
session = requests.Session()

### Special Criteria

**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

*This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration.*


In [9]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"], 
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)

Got 25 ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions
0,4192086697717648,2025-05-05,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-05,2025-05-05,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],434190399773378,Enseignement-Apprentissage,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 36}]",NaN
1,566436209821750,2025-05-05,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-05,2025-05-06,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],355127741015217,Real Team,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 11}]",NaN
2,543784622115975,2025-05-03,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-04,2025-05-04,"[{'country': 'FI', 'age_gender_breakdowns': [{...",[en],100855685416689,Ale Memes esse é o foi,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2}]",NaN
3,569384776193327,2025-05-03,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-03,2025-05-06,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],109194464742841,UsbonGarrh,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Austria', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 181}]",NaN
4,1780320716164479,2025-05-02,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-02,2025-05-03,"[{'country': 'FI', 'age_gender_breakdowns': [{...",[en],105870398554650,Michael EM,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6}]",NaN
5,708779748236211,2025-04-30,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-30,2025-05-01,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],105774929142171,Malucus,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 9}]",NaN
6,1225703642407613,2025-04-30,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-30,2025-05-03,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],102242762540264,Deus é fiel,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 75}]",NaN
7,990332916558558,2025-04-27,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-27,2025-04-29,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],104310028613716,Hustle Nation Team Tubombeko,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 1}]",NaN
8,1256665679256719,2025-07-31,[Login to get free summons😍\nGift Code: vip888🎁],[itunes.apple.com],[🎮 Join now and get 1000 summons FREE!],2025-08-01,2025-08-03,"[{'country': 'DE', 'age_gender_breakdowns': [{...",NaN,126004137251949,Courage Evolution·ⅷ,"[facebook, instagram, audience_network, messen...","[21, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 8}, {'key': 'GB', 'val...",NaN
9,2029047231250712,2025-11-16,[You’re not crazy. You’re just not getting the...,[nas.io],"[Stop guessing. Start Healing, naturally!, Sto...",202

In [10]:
# Checking type of ad in response
political_fields = ["bylines", "currency", "delivery_by_region", "demographic_distribution", "estimated_audience_size", "impressions", "spend"]
political_ads = 0
other_ads = 0
for ad in response["data"]:
    has_political_fields = False
    for k in ad.keys():
        if k in political_fields:
            has_political_fields = True
            break
    if has_political_fields:
        political_ads += 1
    else:
        other_ads += 1

print("Total political ads:", political_ads)
print("Total other categories ads:", other_ads)

Total political ads: 0
Total other categories ads: 25


In [11]:
# EMPLOYMENT_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "EMPLOYMENT_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in EMPLOYMENT_ADS in a single request")
    print(response)
else:
    print("No data found in EMPLOYMENT_ADS")
    print(response)

Got 25 in EMPLOYMENT_ADS in a single request
{'data': [{'id': '871100715574453', 'ad_creation_time': '2025-11-17', 'ad_creative_bodies': ['SAMH is recruiting in Castlemilk, Glasgow 📍\n\nWe’re looking for motivated and resilient Support Workers who are ready to change lives.\n\nWorking at our Hoddam Care Home, you’ll help people with Alcohol Related Brain Damage (ARBD) to live full, meaningful and independent lives.\n\nsamh.org.uk/information/work-with-us'], 'ad_creative_link_captions': ['samh.org.uk/information/work-with-us'], 'ad_creative_link_descriptions': ['SAMH is Scottish Action for Mental Health.'], 'ad_creative_link_titles': ['Ready for a fresh start?'], 'ad_delivery_start_time': '2025-11-18', 'page_id': '304265286356048', 'page_name': 'SAMH', 'publisher_platforms': ['facebook', 'instagram', 'audience_network']}, {'id': '25631983059728274', 'ad_creation_time': '2025-11-17', 'ad_creative_bodies': ['Learn more about Priory. Mental Health Careers - Discover nurse, psychology, ther

In [12]:
# FINANCIAL_PRODUCTS_AND_SERVICES_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "FINANCIAL_PRODUCTS_AND_SERVICES_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in FINANCIAL_PRODUCTS_AND_SERVICES_ADS in a single request")
    print(response)
else:
    print("No data found in FINANCIAL_PRODUCTS_AND_SERVICES_ADS")
    print(response)

Got 25 in FINANCIAL_PRODUCTS_AND_SERVICES_ADS in a single request
{'data': [{'id': '605117375997827', 'ad_creation_time': '2025-11-17', 'ad_creative_bodies': ['Learn more about how clinics compensate sperm donors and what makes the process rewarding both financially and personally'], 'ad_creative_link_captions': ['fintreat.com'], 'ad_creative_link_descriptions': ['Becoming a sperm donor can be a meaningful way to earn money and positively impact people’s lives. Many clinics offer compensation for each contribution, rewarding'], 'ad_creative_link_titles': ['Discover How Sperm Donation Works and Pays'], 'ad_delivery_start_time': '2025-11-17', 'page_id': '703278936212527', 'page_name': 'HealthWay Pro', 'publisher_platforms': ['facebook', 'instagram', 'audience_network', 'messenger', 'threads']}, {'id': '1172121351546925', 'ad_creation_time': '2025-11-17', 'ad_creative_bodies': ['Learn more about how clinics compensate sperm donors and what makes the process rewarding both financially and 

In [13]:
# HOUSING_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "HOUSING_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in HOUSING_ADS in a single request")
    print(response)
else:
    print("No data found in HOUSING_ADS")
    print(response)

Got 25 in HOUSING_ADS in a single request
{'data': [{'id': '1191632569697498', 'ad_creation_time': '2025-11-18', 'ad_creative_bodies': ["If you're a council or housing association tenant living with: \n ✅ Damp or Mould \n ✅ Leaks or Water Damage \n ✅ Broken Heating or Electrics \n ✅ Cracked Walls or Ceilings \n ✅ Pest Infestations \n  \nYou have rights — and we can help you enforce them. \n  \n🔧 Get your home repaired \n 💷 Claim £1,000s in compensation \n ⚠️ No win, no fee. 100% confidential. \n  \n👉 Tap Learn More to check if you qualify in under 60 seconds."], 'ad_creative_link_captions': ['www.housingdisrepairguys.com'], 'ad_creative_link_descriptions': ['Answer a quick checklist to find out if you qualify.'], 'ad_creative_link_titles': ['Act Now – Get Repairs & £1,000s in Compensation!'], 'ad_delivery_start_time': '2025-11-18', 'page_id': '1173281506023298', 'page_name': 'Claims Helpline', 'publisher_platforms': ['facebook', 'instagram', 'audience_network', 'messenger', 'threads']}

In [14]:
# POLITICAL_AND_ISSUE_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in POLITICAL_AND_ISSUE_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in POLITICAL_AND_ISSUE_ADS")
    print(response)

Got 25 ads in POLITICAL_AND_ISSUE_ADS in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,age_country_gender_reach_breakdown,bylines,currency,estimated_audience_size,...,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location,delivery_by_region,demographic_distribution,ad_delivery_stop_time,ad_creative_link_descriptions
0,820201220997208,2025-11-17,"[Every day, Sophie feels the pressure – at hom...",[railwaychildren.org.uk],"[This Christmas, children like Sophie can’t wa...",2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",Railway Children,GBP,"{'lower_bound': '100001', 'upper_bound': '5000...",...,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 23}]",NaN,NaN,NaN,NaN
1,2304576016640979,2025-11-17,[Health is a major issue on doorsteps in Galas...,[survey.labour.org.uk],[Your views matter],2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...","Midlothian South, Tweeddale and Lauderdale Con...",GBP,"{'lower_bound': '10001', 'upper_bound': '50000'}",...,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Galashiels, United Kingdom', 'num_o...","[{'key': 'GB', 'value': 310}]","[{'percentage': '1', 'region': 'Scotland'}]","[{'percentage': '0.032362', 'age': '18-24', 'g...",NaN,NaN
2,1471437670627318,2025-11-17,[Health is a major issue on doorsteps in Galas...,[survey.labour.org.uk],[Your views matter],2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...","Midlothian South, Tweeddale and Lauderdale Con...",GBP,"{'lower_bound': '10001', 'upper_bound': '50000'}",...,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Peebles, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 131}]","[{'percentage': '1', 'region': 'Scotland'}]","[{'percentage': '0.022901', 'age': '18-24', 'g...",2025-11-17,NaN
3,2355445218205574,2025-11-17,[✈️🇪🇸 Travelling to Spain in 2025? Read this b...,[healthplanspain.com],[Schengen Entry Checks in Spain 2025 - Guide f...,2025-11-17,"[{'country': 'IE', 'age_gender_breakdowns': [{...",NaN,EUR,NaN,...,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[20, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 88}, {'key': 'GB', 'va...","[{'percentage': '6.3E-5', 'region': ''Asir Reg...","[{'percentage': '0.035656', 'age': '18-24', 'g...",2025-11-18,[Stricter border controls are now in force in ...
4,1583691626406370,2025-11-17,NaN,"[instagram.com, instagram.com, instagram.com, ...",[Médecins Sans Frontières / Doctors Without Bo...,2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",Médecins Sans Frontières / Doctors Without Bor...,GBP,{'lower_bound': '1000001'},...,[instagram],"{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 1211}]","[{'percentage': '0.837883', 'region': 'England...","[{'percentage': '0.257903', 'age': '18-24', 'g...",NaN,NaN
5,1158136649632381,2025-11-17,[Action For Humanity is on the ground in Yemen...,[actionforhumanity.org],"[Voices From Yemen, Voices From Yemen]",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",Action For Humanity,GBP,"{'lower_bound': '100001', 'upper_bound': '5000...",...,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[20, 55]",All,"[{'name': 'West Bromwich, United Kingdom', 'nu...","[{'key': 'GB', 'value': 35}]",NaN,NaN,NaN,"[Action For Humanity, Action For Humanity]"
6,766996016361133,2025-11-17,[Action For Humanity is on the ground in Yemen...,[actionforhumanity.org],"[Voices From Yemen, Voices From Yemen]",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",Action For Humanity,GBP,"{'lower_bound': '10001', 'upper_bound': '50000'}",...,"[facebook, instagram]","{'lowe

**SC3: Can data from both active and inactive ads be extracted?**

*This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved.*


In [15]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
active_payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ACTIVE",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
inactive_payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "INACTIVE",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
active_response = request_to_api(session, active_payload, api_url)
inactive_response = request_to_api(session, inactive_payload, api_url)
if "data" in active_response:
    print(f"Got {len(active_response['data'])} ACTIVE ads in a single request")
    active_df = pd.DataFrame(active_response["data"])
    display(active_df)
else:
    print("No ACTIVE ads were found")
    print(active_response)
if "data" in inactive_response:
    print(f"Got {len(inactive_response['data'])} INACTIVE ads in a single request")
    inactive_df = pd.DataFrame(inactive_response["data"])
    display(inactive_df)
else:
    print("No INACTIVE ads were found")
    print(inactive_response)


Got 25 ACTIVE ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location
0,667790906185052,2025-10-31,[The Surrey Half Marathon will take place on t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Crawley, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 557}]"
1,668782399418691,2025-10-31,[The Surrey Half Marathon will take place on t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Crawley, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 59}]"
2,794155346789288,2025-10-31,[The Surrey Half Marathon will take place on t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Crawley, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 1019}]"
3,1112181087746067,2025-10-31,[The Surrey Half Marathon will take place on t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Crawley, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 104}]"
4,1163290458618803,2025-10-31,[The Surrey Half Marathon will take place on t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Crawley, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 13}]"
5,1169616888415859,2025-10-30,[The Coventry Half Marathon returns on the 19t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Birmingham, United Kingdom', 'num_o...","[{'key': 'GB', 'value': 204}]"
6,1293431029137185,2025-10-31,[The Surrey Half Marathon will take place on t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Crawley, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 19}]"
7,1323355529118704,2025-10-30,[The Coventry Half Marathon returns on the 19t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Health Research, Raise...",2025-11-17,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],678851558858294,MQ Mental Health Research,"[facebook, instagram, audience_network]","[19, 51]",All,"[{'name': 'Birmingham, United Kingdom', 'num_o...","[{'key': 'GB', 'value': 29}]"
8,1347976580373015,2025-10-31,[The Surrey Half Marathon will take place on t...,[fb.me],"[Run for mental health research, Run for menta...","[Raise Money For Mental Heal

Got 25 INACTIVE ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions
0,4192086697717648,2025-05-05,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-05,2025-05-05,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],434190399773378,Enseignement-Apprentissage,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 36}]",NaN
1,566436209821750,2025-05-05,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-05,2025-05-06,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],355127741015217,Real Team,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 11}]",NaN
2,543784622115975,2025-05-03,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-04,2025-05-04,"[{'country': 'FI', 'age_gender_breakdowns': [{...",[en],100855685416689,Ale Memes esse é o foi,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2}]",NaN
3,569384776193327,2025-05-03,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-03,2025-05-06,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],109194464742841,UsbonGarrh,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Austria', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 181}]",NaN
4,1780320716164479,2025-05-02,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-02,2025-05-03,"[{'country': 'FI', 'age_gender_breakdowns': [{...",[en],105870398554650,Michael EM,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6}]",NaN
5,708779748236211,2025-04-30,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-30,2025-05-01,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],105774929142171,Malucus,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 9}]",NaN
6,1225703642407613,2025-04-30,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-30,2025-05-03,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],102242762540264,Deus é fiel,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 75}]",NaN
7,990332916558558,2025-04-27,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-27,2025-04-29,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],104310028613716,Hustle Nation Team Tubombeko,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 1}]",NaN
8,1256665679256719,2025-07-31,[Login to get free summons😍\nGift Code: vip888🎁],[itunes.apple.com],[🎮 Join now and get 1000 summons FREE!],2025-08-01,2025-08-03,"[{'country': 'DE', 'age_gender_breakdowns': [{...",NaN,126004137251949,Courage Evolution·ⅷ,"[facebook, instagram, audience_network, messen...","[21, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 8}, {'key': 'GB', 'val...",NaN
9,2029047231250712,2025-11-16,[You’re not crazy. You’re just not getting the...,[nas.io],"[Stop guessing. Start Healing, naturally!, Sto...",202

### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.*


In [16]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creative_bodies,ad_creative_link_descriptions,ad_creative_link_titles,bylines,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_creative_bodies,ad_creative_link_titles,page_id,page_name,ad_creative_link_descriptions
0,4192086697717648,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],434190399773378,Enseignement-Apprentissage,NaN
1,566436209821750,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],355127741015217,Real Team,NaN
2,543784622115975,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],100855685416689,Ale Memes esse é o foi,NaN
3,569384776193327,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],109194464742841,UsbonGarrh,NaN
4,1780320716164479,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],105870398554650,Michael EM,NaN
5,708779748236211,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],105774929142171,Malucus,NaN
6,1225703642407613,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],102242762540264,Deus é fiel,NaN
7,990332916558558,"[Tired of dealing with thick, yellow, brittle ...",[Take control of your nail health today],104310028613716,Hustle Nation Team Tubombeko,NaN
8,1256665679256719,[Login to get free summons😍\nGift Code: vip888🎁],[🎮 Join now and get 1000 summons FREE!],126004137251949,Courage Evolution·ⅷ,NaN
9,2029047231250712,[You’re not crazy. You’re just not getting the...,"[Stop guessing. Start Healing, naturally!, Sto...",1476126802688462,So You Think You Are Healthy,"[Comment QUIZ to get the free link now!, Comme..."


In [17]:
if response["data"]:
    URL_PATTERN = r"https?://\S+"
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        columns={},
        checks=pa.Check(
            lambda df: (
                df.select_dtypes(include=['object'])
                .apply(lambda s: s.str.contains(URL_PATTERN).any(), axis=0)
                .sum() == 0
            ),
            name="DirectResponse"
        ),
        strict=False 
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: No links found in any ad creative or authorship column.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Links are present in ad creatives or authorship columns.")
        print(err)

✅ Validation Successful: No links found in any ad creative or authorship column.


**OC5: Can data from an individual ad be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier.*



Ad Library link: [AD ID](https://www.facebook.com/ads/library/?id=794155346789288)

In [18]:
# Use this cell to develop an example where a request for ads from a campaign is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
ad_id = '794155346789288' # SET VALUE HERE
fields="id"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": ad_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_ad = False
    for ad in response["data"]:
        if ad["id"] == ad_id:
            found_ad = True
            break
    if found_ad:
        print(f"Found ad {ad_id} in response")
        print(response)
    else:
        print(f"Ad {ad_id} not found in response")
        print(response)
else:
    print("No data were found in response")
    print(response)

Ad 794155346789288 not found in response
{'data': []}


In [19]:
ad_id = '25547504004847190' # SET AD ID
fields="id"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_page_ids": ad_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_ad = False
    for ad in response["data"]:
        if ad["id"] == ad_id:
            found_ad = True
            break
    if found_ad:
        print(f"Found ad {ad_id} in response")
        print(response)
    else:
        print(f"Ad {ad_id} not found in response")
        print(response)
else:
    print("No data were found in response")
    print(response)

Ad 25547504004847190 not found in response
{'data': []}


**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser.*


In [20]:
# Use this cell to develop an example where a request for ads from an advertiser is made.
# Please leave the result as the output of this cell.
advertiser = '434190399773378' # SET VALUE
fields="id,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_page_ids": advertiser,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_advertiser = False
    other_advertisers = 0
    for ad in response["data"]:
        if ad["page_id"] == advertiser:
            found_advertiser = True
        else:
            other_advertisers += 1
    if found_advertiser and other_advertisers == 0:
        print(f"Only ads from advertiser {advertiser} were found in response")
    elif found_advertiser and other_advertisers > 0:
        print(f"Found ads from advertiser {advertiser} and {other_advertisers} others in response")
    else:
        print(f"Advertiser {advertiser} not found in response")
else:
    print("No data were found in response")
    print(response)

Only ads from advertiser 434190399773378 were found in response


**OC7: Can ad data be retrieved from the platform using search terms?**

*This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data.*


In [21]:
# Use this cell to develop an example where a request for ads is made using a search term.
# Please leave the result as the output of this cell.
search_term = "health" # SET THE VALUE HERE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": search_term,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request using the search term {search_term}")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request using the search term health


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions
0,4192086697717648,2025-05-05,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-05,2025-05-05,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],434190399773378,Enseignement-Apprentissage,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 36}]",NaN
1,566436209821750,2025-05-05,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-05,2025-05-06,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],355127741015217,Real Team,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 11}]",NaN
2,543784622115975,2025-05-03,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-04,2025-05-04,"[{'country': 'FI', 'age_gender_breakdowns': [{...",[en],100855685416689,Ale Memes esse é o foi,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2}]",NaN
3,569384776193327,2025-05-03,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-03,2025-05-06,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],109194464742841,UsbonGarrh,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Austria', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 181}]",NaN
4,1780320716164479,2025-05-02,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-05-02,2025-05-03,"[{'country': 'FI', 'age_gender_breakdowns': [{...",[en],105870398554650,Michael EM,"[facebook, audience_network]","[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6}]",NaN
5,708779748236211,2025-04-30,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-30,2025-05-01,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],105774929142171,Malucus,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 9}]",NaN
6,1225703642407613,2025-04-30,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-30,2025-05-03,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],102242762540264,Deus é fiel,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 75}]",NaN
7,990332916558558,2025-04-27,"[Tired of dealing with thick, yellow, brittle ...",[emscomfy.com],[Take control of your nail health today],2025-04-27,2025-04-29,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],104310028613716,Hustle Nation Team Tubombeko,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 1}]",NaN
8,1256665679256719,2025-07-31,[Login to get free summons😍\nGift Code: vip888🎁],[itunes.apple.com],[🎮 Join now and get 1000 summons FREE!],2025-08-01,2025-08-03,"[{'country': 'DE', 'age_gender_breakdowns': [{...",NaN,126004137251949,Courage Evolution·ⅷ,"[facebook, instagram, audience_network, messen...","[21, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 8}, {'key': 'GB', 'val...",NaN
9,2029047231250712,2025-11-16,[You’re not crazy. You’re just not getting the...,[nas.io],"[Stop guessing. Start Healing, naturally!, Sto...",202

**OC8: Does the platform use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [22]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time,currency"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time
0,4192086697717648,2025-05-05,2025-05-05,2025-05-05
1,566436209821750,2025-05-05,2025-05-05,2025-05-06
2,543784622115975,2025-05-03,2025-05-04,2025-05-04
3,569384776193327,2025-05-03,2025-05-03,2025-05-06
4,1780320716164479,2025-05-02,2025-05-02,2025-05-03
5,708779748236211,2025-04-30,2025-04-30,2025-05-01
6,1225703642407613,2025-04-30,2025-04-30,2025-05-03
7,990332916558558,2025-04-27,2025-04-27,2025-04-29
8,1256665679256719,2025-07-31,2025-08-01,2025-08-03
9,2029047231250712,2025-11-16,2025-11-16,2025-11-17


In [23]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    df["ad_creation_time"] = pd.to_datetime( df["ad_creation_time"])
    df["ad_delivery_start_time"] = pd.to_datetime( df["ad_delivery_start_time"])
    df["ad_delivery_stop_time"] = pd.to_datetime( df["ad_delivery_stop_time"])
    schema = pa.DataFrameSchema(
        {
            "ad_creation_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad creation time"
            ),
            "ad_delivery_start_time": pa.Column(
                pa.DateTime, 
                nullable=False, 
                title="Ad delivery start time"
            ),
            "ad_delivery_stop_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad delivery stop time"
            ),
            "currency": pa.Column(
                pa.String,
                pa.Check.equal_to("GBP"), 
                nullable=False, 
                title="Currency"
            ),
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Data is locale-neutral represented.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Data is not locale-neutral represented.")
        print(err)

❌ Validation Failed: Data is not locale-neutral represented.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'currency' not in dataframe. Columns in dataframe: ['id', 'ad_creation_time', 'ad_delivery_start_time', 'ad_delivery_stop_time']"
            }
        ]
    }
}


In [24]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time,currency"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_creation_time,ad_delivery_start_time,currency,ad_delivery_stop_time
0,820201220997208,2025-11-17,2025-11-18,GBP,NaN
1,2304576016640979,2025-11-17,2025-11-17,GBP,NaN
2,1471437670627318,2025-11-17,2025-11-17,GBP,2025-11-17
3,2355445218205574,2025-11-17,2025-11-17,EUR,2025-11-18
4,1583691626406370,2025-11-17,2025-11-17,GBP,NaN
5,1158136649632381,2025-11-17,2025-11-17,GBP,NaN
6,766996016361133,2025-11-17,2025-11-17,GBP,NaN
7,1506911813892989,2025-11-17,2025-11-17,GBP,NaN
8,4072423809674987,2025-11-17,2025-11-17,CHF,2025-11-18
9,1201738788490856,2025-11-17,2025-11-17,GBP,NaN


In [25]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    df["ad_creation_time"] = pd.to_datetime( df["ad_creation_time"])
    df["ad_delivery_start_time"] = pd.to_datetime( df["ad_delivery_start_time"])
    df["ad_delivery_stop_time"] = pd.to_datetime( df["ad_delivery_stop_time"])
    schema = pa.DataFrameSchema(
        {
            "ad_creation_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad creation time"
            ),
            "ad_delivery_start_time": pa.Column(
                pa.DateTime, 
                nullable=False, 
                title="Ad delivery start time"
            ),
            "ad_delivery_stop_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad delivery stop time"
            ),
            "currency": pa.Column(
                pa.String,
                pa.Check.equal_to("GBP"), 
                nullable=False, 
                title="Currency"
            ),
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Data is locale-neutral represented.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Data is not locale-neutral represented.")
        print(err)

❌ Validation Failed: Data is not locale-neutral represented.
{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "currency",
                "check": "equal_to(GBP)",
                "error": "Column 'currency' failed element-wise validator number 0: equal_to(GBP) failure cases: EUR, CHF, UAH, UAH"
            }
        ]
    }
}


### COMPLETENESS

**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

*This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved.*



In [28]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "*" # SET VALUE
fields="id,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,page_id,page_name
0,2166638110526249,865055823360582,Shoping House
1,1163464415972432,891656217359413,Sumono Shop
2,1604973074260846,371478356677753,Josef Vitzthum
3,1374939944342996,105920594096267,Вакум хаалга цонх засвар / vakum tsonkh zasvar...
4,3193648980809479,216707358190376,Arttaib
5,823238210316661,102854402460581,FiberOne Boats
6,1483626042723309,103858342483163,Zee Big Apple Motors
7,1530792554727145,103858342483163,Zee Big Apple Motors
8,1369911991288823,103858342483163,Zee Big Apple Motors
9,1106852878283096,578775288662683,Express Moving Inc.


In [29]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        {
            "page_id": pa.Column(
                pa.String, 
                nullable=False,  
                title="Page Id"
            ),
            "page_name": pa.Column(
                pa.String, 
                nullable=False,   
                title="Page Name"
            ),
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Advertisers information columns are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing advertisers information.")
        print(err)

✅ Validation Successful: Advertisers information columns are present and non-null.


**OC10: Does the platform provide data on the funders who paid for ads?**

*This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved.*


In [35]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,bylines"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id
0,4192086697717648
1,566436209821750
2,543784622115975
3,569384776193327
4,1780320716164479
5,708779748236211
6,1225703642407613
7,990332916558558
8,1256665679256719
9,2029047231250712


In [36]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        {
            "bylines": pa.Column(
                pa.String, 
                nullable=False,  
                title="Bylines"
            )
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Funders information columns are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing funders information.")
        print(err)

❌ Validation Failed: Missing funders information.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'bylines' not in dataframe. Columns in dataframe: ['id']"
            }
        ]
    }
}


In [33]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,bylines"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,bylines
0,820201220997208,Railway Children
1,2304576016640979,"Midlothian South, Tweeddale and Lauderdale Con..."
2,1471437670627318,"Midlothian South, Tweeddale and Lauderdale Con..."
3,2355445218205574,NaN
4,1583691626406370,Médecins Sans Frontières / Doctors Without Bor...
5,1158136649632381,Action For Humanity
6,766996016361133,Action For Humanity
7,1506911813892989,Action For Humanity
8,4072423809674987,NaN
9,1201738788490856,Re-Elect Christina McAnea for General Secretary


In [34]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        {
            "bylines": pa.Column(
                pa.String, 
                nullable=False,  
                title="Bylines"
            )
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Funders information columns are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing funders information.")
        print(err)

❌ Validation Failed: Missing funders information.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "bylines",
                "check": "not_nullable",
                "error": "non-nullable series 'bylines' contains null values:3     NaN8     NaN10    NaN11    NaN12    NaN13    NaN15    NaN19    NaNName: bylines, dtype: object"
            }
        ]
    }
}


**OC11: Does the platform provide data on the period during which ads were served?**

*This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period*

In [37]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,ad_delivery_start_time,ad_delivery_stop_time"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "INACTIVE",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_delivery_start_time,ad_delivery_stop_time
0,4192086697717648,2025-05-05,2025-05-05
1,566436209821750,2025-05-05,2025-05-06
2,543784622115975,2025-05-04,2025-05-04
3,569384776193327,2025-05-03,2025-05-06
4,1780320716164479,2025-05-02,2025-05-03
5,708779748236211,2025-04-30,2025-05-01
6,1225703642407613,2025-04-30,2025-05-03
7,990332916558558,2025-04-27,2025-04-29
8,1256665679256719,2025-08-01,2025-08-03
9,2029047231250712,2025-11-16,2025-11-17


In [38]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
    df["ad_delivery_stop_time"] = pd.to_datetime(df["ad_delivery_stop_time"])
    schema = pa.DataFrameSchema(
        {
            "ad_delivery_start_time": pa.Column(
                pa.DateTime, 
                nullable=False,  
                title="Ad delivery start time"
            ),
            "ad_delivery_stop_time": pa.Column(
                pa.DateTime, 
                nullable=False,  
                title="Ad delivery stop time"
            )
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Ads' run period dates are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads' run period dates.")
        print(err)


✅ Validation Successful: Ads' run period dates are present and non-null.


**OC12: Does the platform provide data on user engagement with ads?**

*This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign.*


There's no field to retrieve user interactions. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

*This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present.*


There's no field that indicates if a advertiser is verified. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

### COMPLIANCE

**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

*This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented.*



In [58]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "*" # SET VALUE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            'search_page_ids' : ['103858342483163'],
            "search_terms": query,
            "unmask_removed_content": True # unmask set to TRUE
        }
response = request_to_api(session, payload, api_url)
df = pd.DataFrame(response["data"])
display(df)

,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location
0,1483626042723309,2025-11-02,[🚗 For true gearheads only.\nThe CISON Small-b...,[globalshopspot.com],"[🛒 2,519 sold in 24h – grab yours now!]",[🔥BLACK FRIDAY BLOWOUT: Save 49%],2025-11-08,2025-11-08,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],103858342483163,Zee Big Apple Motors,"[facebook, instagram, audience_network, messen...","[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 9}]"
1,1530792554727145,2025-11-01,[🚗 For true gearheads only.\nThe CISON Small-b...,[globalshopspot.com],"[🛒 2,519 sold in 24h – grab yours now!]",[🔥BLACK FRIDAY BLOWOUT: Save 49%],2025-11-02,2025-11-13,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],103858342483163,Zee Big Apple Motors,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 555}]"
2,1369911991288823,2025-10-29,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,2025-11-02,2025-11-08,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],103858342483163,Zee Big Apple Motors,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 3111}]"
3,3436970029791139,2025-11-12,NaN,[facebook.com],NaN,[Zee Big Apple Motors],2025-11-15,2025-11-16,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 25}]"
4,831927239201370,2025-11-01,NaN,[facebook.com],NaN,[Zee Big Apple Motors],2025-11-05,2025-11-08,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 44}]"
5,1262254949275807,2025-10-31,NaN,[facebook.com],NaN,[Zee Big Apple Motors],2025-11-07,2025-11-09,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 20}]"
6,847215507832384,2025-10-28,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,2025-11-01,2025-11-03,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 136}]"


In [ ]:
query = "*" # SET VALUE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            'search_page_ids' : ['103858342483163'],
            "search_terms": query,
            "unmask_removed_content": False # unmask set to FALSE
        }
response = request_to_api(session, payload, api_url)
df = pd.DataFrame(response["data"])
display(df)

,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location
0,1483626042723309,2025-11-02,[🚗 For true gearheads only.\nThe CISON Small-b...,[globalshopspot.com],"[🛒 2,519 sold in 24h – grab yours now!]",[🔥BLACK FRIDAY BLOWOUT: Save 49%],2025-11-08,2025-11-08,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],103858342483163,Zee Big Apple Motors,"[facebook, instagram, audience_network, messen...","[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 9}]"
1,1530792554727145,2025-11-01,[🚗 For true gearheads only.\nThe CISON Small-b...,[globalshopspot.com],"[🛒 2,519 sold in 24h – grab yours now!]",[🔥BLACK FRIDAY BLOWOUT: Save 49%],2025-11-02,2025-11-13,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],103858342483163,Zee Big Apple Motors,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 555}]"
2,1369911991288823,2025-10-29,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,2025-11-02,2025-11-08,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],103858342483163,Zee Big Apple Motors,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 3111}]"
3,3436970029791139,2025-11-12,NaN,[facebook.com],NaN,[Zee Big Apple Motors],2025-11-15,2025-11-16,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 25}]"
4,831927239201370,2025-11-01,NaN,[facebook.com],NaN,[Zee Big Apple Motors],2025-11-05,2025-11-08,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 44}]"
5,1262254949275807,2025-10-31,NaN,[facebook.com],NaN,[Zee Big Apple Motors],2025-11-07,2025-11-09,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 20}]"
6,847215507832384,2025-10-28,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,[This ad was run by an account or Page we late...,2025-11-01,2025-11-03,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,103858342483163,Zee Big Apple Motors,[facebook],"[25, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 136}]"


**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

*This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production.*
 


There's no field that indicates if an ad was created using AI. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

### CONSISTENCY

**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

*This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included.*


In [60]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
page_id = "678851558858294" # SET VALUE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_page_ids": page_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
print(response["data"][0])

{'id': '667790906185052', 'ad_creation_time': '2025-10-31', 'ad_creative_bodies': ['The Surrey Half Marathon will take place on the 22nd March 2026. Sign up today and run for mental health research!', 'The Surrey Half Marathon will take place on the 22nd March 2026. Sign up today and run for mental health research!'], 'ad_creative_link_captions': ['fb.me'], 'ad_creative_link_descriptions': ['Run for mental health research', 'Run for mental health research'], 'ad_creative_link_titles': ['Raise Money For Mental Health Research', 'Raise Money For Mental Health Research'], 'ad_delivery_start_time': '2025-11-17', 'age_country_gender_reach_breakdown': [{'country': 'GB', 'age_gender_breakdowns': [{'age_range': '18-24', 'male': 50, 'female': 43, 'unknown': 1}, {'age_range': '25-34', 'male': 109, 'female': 65, 'unknown': 1}, {'age_range': '35-44', 'male': 133, 'female': 64, 'unknown': 2}, {'age_range': '45-54', 'male': 59, 'female': 29, 'unknown': 3}]}], 'languages': ['en'], 'page_id': '6788515

**OC24: Are the results returned by the platform consistently reproducible?**


*This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results.*

Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [69]:
# Use this cell, or more, to develop a process that collects ads more than one time using the same request parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:
total_runs = 5 
for idx in range(total_runs):
    sleep(10)
    index = idx + 1
    timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S-%f")
    filename = f"uk-meta-ads-question-24-run-{index}-{timestamp}"

    fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
    api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
    query = "health" # SET VALUE
    payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
    response = request_to_api(session, payload, api_url)

    outfile = os.path.join(FILEPATH, f"{filename}.json")

    with open(outfile, "w") as fout:
        json.dump(response, fout)

In [70]:
# Read files here to compare
pattern = os.path.join(FILEPATH, "uk-meta-ads-question-24*")
files = sorted(glob.glob(pattern))
all_files = []
for idx, file in enumerate(files):
    with open(file) as fin:
        data = json.load(fin)
    df = pd.DataFrame(data["data"])
    df["filename"] = os.path.basename(file)
    all_files.append(df)
df_complete = pd.concat(all_files, ignore_index=True)
df_agg = df_complete.groupby("id").agg(total_occurance=("id", "count"),first_occurance_file=("filename", "min"), last_occurance_file=("filename", "max"),start_delivery_date_max=("ad_delivery_start_time", "max"))
display(df_agg)
    

,total_occurance,first_occurance_file,last_occurance_file,start_delivery_date_max
id,,,,
1118856549901694,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2024-12-25
1149767699841006,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2024-12-23
1198154008406828,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2024-12-19
1225703642407613,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2025-04-30
1238312724100050,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2024-12-23
1256665679256719,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2025-08-01
1266941774632571,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2024-12-23
1510983603181878,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2025-10-22
1645469656179516,5,uk-meta-ads-question-24-run-1-2025-11-18T14-55...,uk-meta-ads-question-24-run-5-2025-11-18T14-56...,2024-12-25


**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

*This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions.*

Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [102]:
# Use this cell, or more, to develop a process that collects ads more than one time using different parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:

#FEEL FREE TO CHANGE PARAMETERS
parameters = [{"ad_delivery_date_min": "2025-11-01"}, 
                {"bylines": ["Labour Future"]}, 
                {"delivery_by_region": ["England"]}, 
                {"estimated_audience_size_min":1000}, 
                {"languages": ["en"]}]
for idx, param in enumerate(parameters):
    sleep(1)
    index = idx + 1 
    timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S-%f")
    filename = f"uk-meta-ads-question-25-run-{index}-{list(param.keys())[0]}-{timestamp}"
    
    fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
    api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
    query = "health" # Feel free to change value
    payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "INACTIVE",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
    payload.update(param)
    response = request_to_api(session, payload, api_url)

    with open(f"{FILEPATH}/{filename}.json", "w") as fout:
        json.dump(response, fout)

In [104]:
# ad_delivery_date_min > 2025-11-01
filename = "uk-meta-ads-question-25-run-1-ad_delivery_date_min-2025-11-18T17-22-56-709498.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)
df = pd.DataFrame(data["data"])
df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
df["ad_delivery_stop_time"] = pd.to_datetime(df["ad_delivery_stop_time"])
schema = pa.DataFrameSchema(
    {
        "ad_delivery_stop_time": pa.Column(
            pa.DateTime, 
            checks=pa.Check.greater_than_or_equal_to("2025-11-01"),
            nullable=False,  
            title="Ad delivery start time"
        )
    },
    strict=False
)
try:
    schema.validate(df, lazy=True)
    print("✅ Validation Successful: All ads ran after the set ad_delivery_date_min parameter.")
except SchemaErrors as err:
    print("❌ Validation Failed: Found ads that ran earlier than the set ad_delivery_date_min parameter")
    print(err)




✅ Validation Successful: All ads ran after the set ad_delivery_date_min parameter.


In [105]:
# bylines: Labour Future
filename = "uk-meta-ads-question-25-run-2-bylines-2025-11-18T17-23-00-143281.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)
if "data" in data:
    df = pd.DataFrame(data["data"])

    schema = pa.DataFrameSchema(
        {
            "bylines": pa.Column(
                checks=pa.Check.equal_to("Labour Future"),
                nullable=False,  
                title="Bylines"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have funders information as set in bylines parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that are not from funders as set in bylines parameter")
        print(err)




✅ Validation Successful: All ads have funders information as set in bylines parameter.


In [106]:
# delivery_by_region: Manchester
def extract_regions(x):
    if isinstance(x, list) and len(x) > 0: 
        return [d['region'] for d in x]
    return x

def check_delivery_region(regions):
    return regions.apply(lambda region_list: "England" in region_list)

filename = "uk-meta-ads-question-25-run-3-delivery_by_region-2025-11-18T17-23-01-844199.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.DataFrame(data["data"])
    df["ad_delivery_regions"] = df["delivery_by_region"].apply(extract_regions)

    schema = pa.DataFrameSchema(
        {
            "ad_delivery_regions": pa.Column(
                checks=pa.Check(check_delivery_region),
                nullable=False,  
                title="Ad delivery regions"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have region information as set in delivery_by_region parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the regions as set in delivery_by_region parameter")
        print(err)





❌ Validation Failed: Found ads that don't have or are not from the regions as set in delivery_by_region parameter
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "ad_delivery_regions",
                "check": "not_nullable",
                "error": "non-nullable series 'ad_delivery_regions' contains null values:1     NaN11    NaN13    NaNName: ad_delivery_regions, dtype: object"
            }
        ]
    }
}


In [ ]:
# estimated_audience_size_min > 500
filename = "uk-meta-ads-question-25-run-4-estimated_audience_size_min-2025-11-18T17-23-04-567713.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.json_normalize(data["data"])
    df["estimated_audience_size.lower_bound"] = df["estimated_audience_size.lower_bound"].astype("int64") # NEEDS TO DECLARE INT64, AS SOME OF THEM CAME AS INT32
   
    schema = pa.DataFrameSchema(
        {
            "estimated_audience_size.lower_bound": pa.Column(
                pa.Int, 
                checks=pa.Check.greater_than_or_equal_to(500),
                nullable=False,  
                title="Estimated audience size - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have audience size greater than the set estimated_audience_size_min parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that have audience size lower than the set estimated_audience_size_min parameter")
        print(err)

✅ Validation Successful: All ads have audience size greater than the set estimated_audience_size_min parameter.


In [109]:
# language: en
filename = "uk-meta-ads-question-25-run-5-languages-2025-11-18T17-23-07-162126.json"
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.DataFrame(data["data"])
    df["languages"] = df["languages"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)


    schema = pa.DataFrameSchema(
        {
            "languages": pa.Column(
                checks=pa.Check.equal_to("en"),
                nullable=False,  
                title="Languages"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have information regarding language and it's the correct language set in language parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the language set in the language parameter")
        print(err)





✅ Validation Successful: All ads have information regarding language and it's the correct language set in language parameter.


### RELEVANCE

**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

*This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges.*



In [111]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True,
            "ad_delivery_date_max": "2025-10-15", # FEEL FREE TO CHANGE DATES
            "ad_delivery_date_min": "2025-10-01" # FEEL FREE TO CHANGE DATES
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)


Got 25 ads in a single request


,id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time
0,1319582746241612,2025-10-01,2025-10-02,2025-10-03
1,1769832226983707,2025-10-08,2025-10-09,2025-10-09
2,2757687637759616,2025-10-08,2025-10-09,2025-10-11
3,3287991251366095,2025-10-08,2025-10-09,2025-10-11
4,707395975221816,2025-10-15,2025-10-15,2025-10-22
5,1823552321597902,2025-10-15,2025-10-15,2025-10-22
6,2221930414885062,2025-10-07,2025-10-07,2025-10-09
7,565344519972786,2025-09-29,2025-10-02,2025-10-03
8,676225511673297,2025-09-29,2025-10-03,2025-10-05
9,777855338400759,2025-09-29,2025-10-04,2025-10-05


In [112]:
df = pd.DataFrame(response["data"])
df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
start_date = datetime(year=2025, month=10, day=1)
end_date = datetime(year=2025, month=10, day=15)

schema = pa.DataFrameSchema({
    "ad_delivery_start_time": pa.Column(
        pa.DateTime,
        pa.Check(lambda s: s.between(start_date, end_date, inclusive="both"), 
                 name="DateRangeCheck"),
        nullable=False
    )
})
try:
    schema.validate(df, lazy=True)
    print("✅ Validation Successful: All ads are between date range.")

except SchemaErrors as err:
    print("❌ Validation Failed: Found ads outside date range.")
    print(err) 

✅ Validation Successful: All ads are between date range.


**OC27: Does the platform allow filtering advertising data by ad category?**

*This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications.*


In [113]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "EMPLOYMENT_ADS", # FEEL FREE TO CHANGE CATEGORY
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in EMPLOYMENT_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in EMPLOYMENT_ADS")
    print(response)

Got 25 in EMPLOYMENT_ADS in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions,ad_delivery_stop_time
0,1548921109481590,2025-11-18,[Learn more about Priory. Mental Health Career...,"[jobs.priorygroup.com, jobs.priorygroup.com, j...","[Find your new role with Priory, Find your new...",2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],5944538380,Indeed,"[facebook, instagram, audience_network, threads]","[18, 65]",All,"[{'name': 'Bristol, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 1}]",NaN,NaN
1,1373133414457420,2025-11-18,[👋 Step into a role at Co-op to work in an org...,"[coop.co.uk, coop.co.uk, coop.co.uk]","[Become a Team Leader, Become a Team Leader, B...",2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],10150110761225377,Co-op,"[facebook, instagram]","[18, 65]",All,"[{'name': 'Sandhurst, United Kingdom', 'num_ob...","[{'key': 'GB', 'value': 2}]","[Find a role near you, Find a role near you, F...",NaN
2,831141929646269,2025-11-18,[👋 Step into a role at Co-op to work in an org...,"[coop.co.uk, coop.co.uk, coop.co.uk]","[Become a Team Leader, Become a Team Leader, B...",2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],10150110761225377,Co-op,"[facebook, instagram]","[18, 65]",All,"[{'name': 'Sandhurst, United Kingdom', 'num_ob...","[{'key': 'GB', 'value': 77}]","[Find a role near you, Find a role near you, F...",NaN
3,1155068040102937,2025-11-18,[👋 Step into a role at Co-op to work in an org...,"[coop.co.uk, coop.co.uk, coop.co.uk]","[Apply now, Apply now, Apply now]",2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],10150110761225377,Co-op,"[facebook, instagram]","[18, 65]",All,"[{'name': 'Central Worthing, Worthing, United ...","[{'key': 'GB', 'value': 11}]","[Find a role near you, Find a role near you, F...",NaN
4,1111750867697594,2025-11-18,[👋 Step into a role at Co-op to work in an org...,"[coop.co.uk, coop.co.uk, coop.co.uk]","[Apply now, Apply now, Apply now]",2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],10150110761225377,Co-op,"[facebook, instagram]","[18, 65]",All,"[{'name': 'Central Worthing, Worthing, United ...","[{'key': 'GB', 'value': 31}]","[Find a role near you, Find a role near you, F...",NaN
5,1555182309136039,2025-11-18,[Ever wondered what makes GSK such a great pla...,[d.hodes.com],[GSK: Your Career in Global Health​],2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],512439245793683,GSK careers,"[facebook, instagram, messenger]","[18, 65]",All,"[{'name': 'Barnard Castle, United Kingdom', 'n...","[{'key': 'GB', 'value': 180}]",NaN,NaN
6,1178485803687227,2025-11-18,[Want to be part of something bigger? 🌍 ​\n\nK...,[gsk.com],[GSK: Your Career in Global Health​],2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],123311187737843,GSK,[instagram],"[18, 65]",All,"[{'name': 'Barnard Castle, United Kingdom', 'n...","[{'key': 'GB', 'value': 15}]",NaN,NaN
7,708536035174709,2025-11-18,[“I changed careers and never looked back.” – ...,[fb.com],[“I changed careers and never looked back.” – ...,2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],308731479193928,Carr Gomm,[facebook],"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 304}]",NaN,2025-11-18
8,1527697131989597,2025-11-18,[❗️Brand New Vacancy : Therapeutic Lead❗️\n\nW...,NaN,NaN,2025-11-18,"[{'country': 'GB', 'age_gender_breakdowns': [{...",[en],1497175786981822,FEVACA,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 94}]",NaN,NaN
9,1900063050947561,2025-11-18,"[Every day, our people Go Beyond — sparking cr...",[careers.dsm-firmenich.com],"[Ready to go beyond?,

**OC28: Does the platform allow filtering advertising data by geographic location?**

*This item assesses whether the ad repository allows filtering data by one or more subnational geographic locations where the ads were served. The assessment should test queries with location filters to confirm that results match the specified areas.*

In [8]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "delivery_by_region": ["Wales"], # FEEL FREE TO CHANGE REGION
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

No data found
{'error': {'message': 'An unknown error has occurred.', 'type': 'OAuthException', 'code': 1, 'fbtrace_id': 'AuviBPiPclMfb5Q7v_pINV-'}}


In [9]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "delivery_by_region": ["Wales"], # FEEL FREE TO CHANGE REGION
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,delivery_by_region
0,2000140754078974,NaN
1,873024729013840,"[{'percentage': '0.835799', 'region': 'England..."
2,1389542052586614,"[{'percentage': '0.835052', 'region': 'England..."
3,1231020129047078,NaN
4,1089813986421302,"[{'percentage': '0.824675', 'region': 'England..."
5,1180576226824789,"[{'percentage': '0.866972', 'region': 'England..."
6,2025553841632824,"[{'percentage': '0.76129', 'region': 'England'..."
7,843582031655873,"[{'percentage': '0.768557', 'region': 'England..."
8,846774491643583,"[{'percentage': '0.001043', 'region': 'Abia St..."
9,2346315129155988,"[{'percentage': '0.026403', 'region': 'Abkhazi..."


In [10]:
# delivery_by_region: Dublin
def extract_regions(x):
    if isinstance(x, list) and len(x) > 0:
        return [d['region'] for d in x]
    return x

def check_delivery_region(regions):
    return regions.apply(lambda region_list: "Wales" in region_list)

if "data" in response:
    df = pd.DataFrame(response["data"])
    df["ad_delivery_regions"] = df["delivery_by_region"].apply(extract_regions)

    schema = pa.DataFrameSchema(
        {
            "ad_delivery_regions": pa.Column(
                checks=pa.Check(check_delivery_region),
                nullable=False,  
                title="Ad delivery regions"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads came from the region set in the geographic location filter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the region set in the geographic location filter.")
        print(err)





❌ Validation Failed: Found ads that don't have or are not from the region set in the geographic location filter.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "ad_delivery_regions",
                "check": "not_nullable",
                "error": "non-nullable series 'ad_delivery_regions' contains null values:0     NaN3     NaN17    NaNName: ad_delivery_regions, dtype: object"
            }
        ]
    }
}


In [13]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "delivery_by_region": ["London"], # FEEL FREE TO CHANGE REGION
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

No data found
{'error': {'message': 'An unknown error has occurred.', 'type': 'OAuthException', 'code': 1, 'fbtrace_id': 'Akeq-U43zWL3Mf9iqUWDvHd'}}


### ACCURACY

**OC29: Does the platform provide age and gender data on the audiences of ads?**

*This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported*

In [127]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,age_country_gender_reach_breakdown,demographic_distribution,target_ages,target_gender"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,age_country_gender_reach_breakdown,target_ages,target_gender
0,4192086697717648,"[{'country': 'FR', 'age_gender_breakdowns': [{...","[18, 65]",All
1,566436209821750,"[{'country': 'FR', 'age_gender_breakdowns': [{...","[18, 65]",All
2,543784622115975,"[{'country': 'FI', 'age_gender_breakdowns': [{...","[18, 65]",All
3,569384776193327,"[{'country': 'AT', 'age_gender_breakdowns': [{...","[18, 65]",All
4,1780320716164479,"[{'country': 'FI', 'age_gender_breakdowns': [{...","[18, 65]",All
5,708779748236211,"[{'country': 'PT', 'age_gender_breakdowns': [{...","[18, 65]",All
6,1225703642407613,"[{'country': 'PT', 'age_gender_breakdowns': [{...","[18, 65]",All
7,990332916558558,"[{'country': 'PT', 'age_gender_breakdowns': [{...","[18, 65]",All
8,1256665679256719,"[{'country': 'DE', 'age_gender_breakdowns': [{...","[21, 65]",All
9,2029047231250712,"[{'country': 'GB', 'age_gender_breakdowns': [{...","[18, 65]",All


In [128]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "age_country_gender_reach_breakdown": pa.Column( 
                nullable=False,  
                title="Age Country Gender Reach Breakdown"
            ),
            "target_ages": pa.Column(
                nullable=False,  
                title="Target Ages"
            ),
            "target_gender": pa.Column(
                nullable=False,  
                title="Target Ages"
            ),
            "demographic_distribution": pa.Column(
                nullable=False,  
                title="Target Ages"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have information about age and gender about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads information about age and gender.")
        print(err)


❌ Validation Failed: Missing ads information about age and gender.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'demographic_distribution' not in dataframe. Columns in dataframe: ['id', 'age_country_gender_reach_breakdown', 'target_ages', 'target_gender']"
            }
        ]
    }
}


In [129]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,age_country_gender_reach_breakdown,demographic_distribution,target_ages,target_gender"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,age_country_gender_reach_breakdown,demographic_distribution,target_ages,target_gender
0,2676039509409546,"[{'country': 'ES', 'age_gender_breakdowns': [{...","[{'percentage': '0.051429', 'age': '25-34', 'g...","[25, 65]",All
1,863253100206091,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,"[18, 65]",All
2,1193966449497472,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,"[18, 65]",All
3,1544865313368139,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,"[18, 65]",All
4,820201220997208,"[{'country': 'GB', 'age_gender_breakdowns': [{...","[{'percentage': '0.037698', 'age': '18-24', 'g...","[18, 65]",All
5,2304576016640979,"[{'country': 'GB', 'age_gender_breakdowns': [{...","[{'percentage': '0.023753', 'age': '18-24', 'g...","[18, 65]",All
6,1471437670627318,"[{'country': 'GB', 'age_gender_breakdowns': [{...","[{'percentage': '0.022901', 'age': '18-24', 'g...","[18, 65]",All
7,2355445218205574,"[{'country': 'IE', 'age_gender_breakdowns': [{...","[{'percentage': '0.035667', 'age': '18-24', 'g...","[20, 65]",All
8,1583691626406370,"[{'country': 'GB', 'age_gender_breakdowns': [{...","[{'percentage': '0.310861', 'age': '18-24', 'g...","[18, 65]",All
9,1158136649632381,"[{'country': 'GB', 'age_gender_breakdowns': [{...",NaN,"[20, 55]",All


In [130]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "age_country_gender_reach_breakdown": pa.Column( 
                nullable=False,  
                title="Age Country Gender Reach Breakdown"
            ),
            "target_ages": pa.Column(
                nullable=False,  
                title="Target Ages"
            ),
            "target_gender": pa.Column(
                nullable=False,  
                title="Target Ages"
            ),
            "demographic_distribution": pa.Column(
                nullable=False,  
                title="Target Ages"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have information about age and gender about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads information about age and gender.")
        print(err)


❌ Validation Failed: Missing ads information about age and gender.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "demographic_distribution",
                "check": "not_nullable",
                "error": "non-nullable series 'demographic_distribution' contains null values:1     NaN2     NaN3     NaN9     NaN10    NaN11    NaN15    NaN17    NaNName: demographic_distribution, dtype: object"
            }
        ]
    }
}


**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

*This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported.*

In [131]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,target_locations,total_reach_by_location
0,4192086697717648,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 36}]"
1,566436209821750,"[{'name': 'France', 'num_obfuscated': 0, 'type...","[{'key': 'EU', 'value': 11}]"
2,543784622115975,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2}]"
3,569384776193327,"[{'name': 'Austria', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 181}]"
4,1780320716164479,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6}]"
5,708779748236211,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 9}]"
6,1225703642407613,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 75}]"
7,990332916558558,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 1}]"
8,1256665679256719,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 8}, {'key': 'GB', 'val..."
9,2029047231250712,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 65}]"


In [132]:
def check_region_granularity(target_locations): 
    subnational_granularity = False
    if isinstance(target_locations, list) and len(target_locations) > 0:
        for location in target_locations:
            locations = location.get("name").split(",") 
            # TO DO: Check if only England/Scotland/Wales/Northern Ireland are left
            if len(locations) > 1:
                subnational_granularity = True
                break
    return subnational_granularity

def validate_granularity_column(s):
    return s.apply(check_region_granularity)

if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "delivery_by_region": pa.Column( 
                nullable=False,  
                title="Delivery by region"
            ),
            "target_locations": pa.Column(
                pa.Object,
                pa.Check(validate_granularity_column, error="Ad lacks required subnational granularity."),
                nullable=False,  
                title="Target location"
            ),
            "total_reach_by_location": pa.Column(
                nullable=False,  
                title="Total reach by location"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have subnational geographic information about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads subnational geographic information.")
        print(err)


❌ Validation Failed: Missing ads subnational geographic information.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'delivery_by_region' not in dataframe. Columns in dataframe: ['id', 'target_locations', 'total_reach_by_location']"
            }
        ]
    },
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "target_locations",
                "check": "Ad lacks required subnational granularity.",
                "error": "Column 'target_locations' failed element-wise validator number 0: Ad lacks required subnational granularity. failure cases: [{'name': 'France', 'num_obfuscated': 0, 'type': 'countries', 'excluded': False}], [{'name': 'France', 'num_obfuscated': 0, 'type': 'countries', 'excluded': False}], [{'name': 'Finland', 'num_obfuscated': 0, 'type'

In [143]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,delivery_by_region,target_locations,total_reach_by_location
0,2676039509409546,"[{'percentage': '0.005714', 'region': 'Aargau'...","[{'name': 'Netherlands', 'num_obfuscated': 0, ...","[{'key': 'EU', 'value': 50}, {'key': 'GB', 'va..."
1,863253100206091,NaN,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 70}]"
2,1193966449497472,NaN,"[{'name': 'London, United Kingdom', 'num_obfus...","[{'key': 'GB', 'value': 59}]"
3,1544865313368139,NaN,"[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 55}]"
4,820201220997208,"[{'percentage': '0.879447', 'region': 'England...","[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 507}]"
5,2304576016640979,"[{'percentage': '1', 'region': 'Scotland'}]","[{'name': 'Galashiels, United Kingdom', 'num_o...","[{'key': 'GB', 'value': 525}]"
6,1471437670627318,"[{'percentage': '1', 'region': 'Scotland'}]","[{'name': 'Peebles, United Kingdom', 'num_obfu...","[{'key': 'GB', 'value': 131}]"
7,2355445218205574,"[{'percentage': '6.2E-5', 'region': ''Asir Reg...","[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 88}, {'key': 'GB', 'va..."
8,1583691626406370,"[{'percentage': '0.835996', 'region': 'England...","[{'name': 'United Kingdom', 'num_obfuscated': ...","[{'key': 'GB', 'value': 1879}]"
9,1158136649632381,NaN,"[{'name': 'West Bromwich, United Kingdom', 'nu...","[{'key': 'GB', 'value': 41}]"


In [144]:
def check_region_granularity(target_locations): 
    subnational_granularity = False
    if isinstance(target_locations, list) and len(target_locations) > 0:
        for location in target_locations:
            locations = location.get("name").split(",") 
            # TO DO: Check if only England/Scotland/Wales/Northern Ireland are left
            if len(locations) > 1:
                subnational_granularity = True
                break
    return subnational_granularity

def validate_granularity_column(s):
    return s.apply(check_region_granularity)

if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "delivery_by_region": pa.Column( 
                nullable=False,  
                title="Delivery by region"
            ),
            "total_reach_by_location": pa.Column(
                nullable=False,  
                title="Total reach by location"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have subnational geographic information about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads subnational geographic information.")
        print(err)


❌ Validation Failed: Missing ads subnational geographic information.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "delivery_by_region",
                "check": "not_nullable",
                "error": "non-nullable series 'delivery_by_region' contains null values:1     NaN2     NaN3     NaN9     NaN10    NaN11    NaN15    NaN17    NaNName: delivery_by_region, dtype: object"
            }
        ]
    }
}


**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

*This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported.*


In [149]:
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,estimated_audience_size,target_ages,target_gender,target_locations"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,target_ages,target_gender,target_locations
0,4192086697717648,"[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type..."
1,566436209821750,"[18, 65]",All,"[{'name': 'France', 'num_obfuscated': 0, 'type..."
2,543784622115975,"[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ..."
3,569384776193327,"[18, 65]",All,"[{'name': 'Austria', 'num_obfuscated': 0, 'typ..."
4,1780320716164479,"[18, 65]",All,"[{'name': 'Finland', 'num_obfuscated': 0, 'typ..."
5,708779748236211,"[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty..."
6,1225703642407613,"[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty..."
7,990332916558558,"[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty..."
8,1256665679256719,"[21, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '..."
9,2029047231250712,"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ..."


In [150]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "target_ages": pa.Column(
                nullable=False,
                title="Target Ages"
            ),
            "target_gender": pa.Column(
                nullable=False,
                title="Target Gender"
            ),
            "target_locations": pa.Column(
                nullable=False,
                title="Target Locations"
            ),
        },
        strict=False
    )

    try:
        schema.validate(df, lazy=True)
        print("✅ OC31: Validation successful – for this sample, all ads include age, gender, and geographic targeting fields defined by the advertiser.")
    except SchemaErrors as err:
        print("❌ OC31: Validation failed – some ads are missing age, gender, or geographic targeting information.")
        print(err)
else:
    print("No data found")
    print(response)


✅ OC31: Validation successful – for this sample, all ads include age, gender, and geographic targeting fields defined by the advertiser.


In [145]:
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,estimated_audience_size,target_ages,target_gender,target_locations"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,target_ages,target_gender,target_locations,estimated_audience_size
0,2676039509409546,"[25, 65]",All,"[{'name': 'Netherlands', 'num_obfuscated': 0, ...",NaN
1,863253100206091,"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...",{'lower_bound': '1000001'}
2,1193966449497472,"[18, 65]",All,"[{'name': 'London, United Kingdom', 'num_obfus...","{'lower_bound': '100001', 'upper_bound': '5000..."
3,1544865313368139,"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...",{'lower_bound': '1000001'}
4,820201220997208,"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...","{'lower_bound': '100001', 'upper_bound': '5000..."
5,2304576016640979,"[18, 65]",All,"[{'name': 'Galashiels, United Kingdom', 'num_o...","{'lower_bound': '10001', 'upper_bound': '50000'}"
6,1471437670627318,"[18, 65]",All,"[{'name': 'Peebles, United Kingdom', 'num_obfu...","{'lower_bound': '10001', 'upper_bound': '50000'}"
7,2355445218205574,"[20, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...",{'lower_bound': '1000001'}
8,1583691626406370,"[18, 65]",All,"[{'name': 'United Kingdom', 'num_obfuscated': ...",{'lower_bound': '1000001'}
9,1158136649632381,"[20, 55]",All,"[{'name': 'West Bromwich, United Kingdom', 'nu...","{'lower_bound': '100001', 'upper_bound': '5000..."


In [147]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "target_ages": pa.Column(
                nullable=False,
                title="Target Ages"
            ),
            "target_gender": pa.Column(
                nullable=False,
                title="Target Gender"
            ),
            "target_locations": pa.Column(
                nullable=False,
                title="Target Locations"
            ),
        },
        strict=False
    )

    try:
        schema.validate(df, lazy=True)
        print("✅ OC31: Validation successful – for this sample, all ads include age, gender, and geographic targeting fields defined by the advertiser.")
    except SchemaErrors as err:
        print("❌ OC31: Validation failed – some ads are missing age, gender, or geographic targeting information.")
        print(err)
else:
    print("No data found")
    print(response)


✅ OC31: Validation successful – for this sample, all ads include age, gender, and geographic targeting fields defined by the advertiser.


**OC32: Does the platform provide granular volume ranges for ad impressions?**

*This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [155]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,impressions"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id
0,4192086697717648
1,566436209821750
2,543784622115975
3,569384776193327
4,1780320716164479
5,708779748236211
6,1225703642407613
7,990332916558558
8,1256665679256719
9,2029047231250712


In [157]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    required_cols = ["impressions.upper_bound", "impressions.lower_bound"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required impressions columns: {missing}")
    df["impressions.upper_bound"]= df["impressions.upper_bound"].astype(int)
    df["percentage"] = round((df["impressions.upper_bound"].astype(int) - df["impressions.lower_bound"].astype(int) ) / df["impressions.upper_bound"].astype(int)  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between impressions values percentage"
            ),
            "impressions.upper_bound": pa.Column(
                nullable=False,  
                title="Impressions - Upper"
            ),
            "impressions.lower_bound": pa.Column(
                nullable=False,  
                title="Impressions - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have impressions granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Impressions granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


ValueError: Missing required impressions columns: ['impressions.upper_bound', 'impressions.lower_bound']

In [158]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,impressions"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id,impressions
0,2676039509409546,"{'lower_bound': '0', 'upper_bound': '999'}"
1,863253100206091,"{'lower_bound': '0', 'upper_bound': '999'}"
2,1193966449497472,"{'lower_bound': '0', 'upper_bound': '999'}"
3,1544865313368139,"{'lower_bound': '0', 'upper_bound': '999'}"
4,820201220997208,"{'lower_bound': '0', 'upper_bound': '999'}"
5,2304576016640979,"{'lower_bound': '0', 'upper_bound': '999'}"
6,1471437670627318,"{'lower_bound': '0', 'upper_bound': '999'}"
7,2355445218205574,"{'lower_bound': '50000', 'upper_bound': '59999'}"
8,1583691626406370,"{'lower_bound': '2000', 'upper_bound': '2999'}"
9,1158136649632381,"{'lower_bound': '0', 'upper_bound': '999'}"


In [159]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    required_cols = ["impressions.upper_bound", "impressions.lower_bound"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required impressions columns: {missing}")
    df["impressions.upper_bound"]= df["impressions.upper_bound"].astype(int)
    df["percentage"] = round((df["impressions.upper_bound"].astype(int) - df["impressions.lower_bound"].astype(int) ) / df["impressions.upper_bound"].astype(int)  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between impressions values percentage"
            ),
            "impressions.upper_bound": pa.Column(
                nullable=False,  
                title="Impressions - Upper"
            ),
            "impressions.lower_bound": pa.Column(
                nullable=False,  
                title="Impressions - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have impressions granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Impressions granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


❌ Validation Failed: Impressions granularity between lower and upper bound are larger than 10% of the upper bound
{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "percentage",
                "check": "less_than_or_equal_to(10)",
                "error": "Column 'percentage' failed element-wise validator number 0: less_than_or_equal_to(10) failure cases: 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 16.7, 33.3, 100.0, 100.0, 100.0, 100.0, 20.0, 100.0, 100.0, 25.0, 100.0, 33.3, 100.0, 16.7, 33.3, 100.0, 25.0"
            }
        ]
    }
}


**OC33: Does the platform provide granular investment ranges for ad spending?**

*This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [160]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,spend"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id
0,4192086697717648
1,566436209821750
2,543784622115975
3,569384776193327
4,1780320716164479
5,708779748236211
6,1225703642407613
7,990332916558558
8,1256665679256719
9,2029047231250712


In [162]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    required_cols = ["spend.upper_bound", "spend.lower_bound"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required spend columns: {missing}")
    df["spend.upper_bound"]= df["spend.upper_bound"].astype(int)
    df["spend.lower_bound"]= df["spend.lower_bound"].astype(int)
    df["percentage"] = round((df["spend.upper_bound"] - df["spend.lower_bound"] ) / df["spend.upper_bound"]  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between spend values percentage"
            ),
            "spend.upper_bound": pa.Column(
                nullable=False,  
                title="Spend - Upper"
            ),
            "spend.lower_bound": pa.Column(
                nullable=False,  
                title="Spend - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have spend granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Spend granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


ValueError: Missing required spend columns: ['spend.upper_bound', 'spend.lower_bound']

In [14]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,spend"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id,spend
0,910738265449473,"{'lower_bound': '0', 'upper_bound': '99'}"
1,1538121610742057,"{'lower_bound': '0', 'upper_bound': '99'}"
2,1233346365282183,"{'lower_bound': '0', 'upper_bound': '99'}"
3,1938522950057502,"{'lower_bound': '0', 'upper_bound': '99'}"
4,4081978308686111,"{'lower_bound': '0', 'upper_bound': '99'}"
5,2000140754078974,"{'lower_bound': '0', 'upper_bound': '99'}"
6,873409422033200,"{'lower_bound': '0', 'upper_bound': '99'}"
7,1059079012937652,"{'lower_bound': '0', 'upper_bound': '99'}"
8,1926937461217290,"{'lower_bound': '0', 'upper_bound': '99'}"
9,1219033433450981,"{'lower_bound': '0', 'upper_bound': '99'}"


In [15]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    required_cols = ["spend.upper_bound", "spend.lower_bound"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required spend columns: {missing}")
    df["spend.upper_bound"]= df["spend.upper_bound"].astype(int)
    df["spend.lower_bound"]= df["spend.lower_bound"].astype(int)
    df["percentage"] = round((df["spend.upper_bound"] - df["spend.lower_bound"] ) / df["spend.upper_bound"]  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between spend values percentage"
            ),
            "spend.upper_bound": pa.Column(
                nullable=False,  
                title="Spend - Upper"
            ),
            "spend.lower_bound": pa.Column(
                nullable=False,  
                title="Spend - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have spend granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Spend granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


❌ Validation Failed: Spend granularity between lower and upper bound are larger than 10% of the upper bound
{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "percentage",
                "check": "less_than_or_equal_to(10)",
                "error": "Column 'percentage' failed element-wise validator number 0: less_than_or_equal_to(10) failure cases: 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0"
            }
        ]
    }
}


In [16]:
df

,id,spend.lower_bound,spend.upper_bound,percentage
0,910738265449473,0,99,100.0
1,1538121610742057,0,99,100.0
2,1233346365282183,0,99,100.0
3,1938522950057502,0,99,100.0
4,4081978308686111,0,99,100.0
5,2000140754078974,0,99,100.0
6,873409422033200,0,99,100.0
7,1059079012937652,0,99,100.0
8,1926937461217290,0,99,100.0
9,1219033433450981,0,99,100.0
